# A2 vs resting-state — clean cross-task re-ID

Tightens the A2 cross-task claim. The original A2 trains the re-ID probe on motor-execution embeddings (related task), so the 'task-orthogonal identity' framing is soft. This experiment uses PhysioNet's resting-state runs (R01 eyes open, R02 eyes closed) as the probe-train data — no shared task structure with motor imagery — and tests on imagery test runs (R12, R14). If the probe still recovers identity at high accuracy, the identity signal is genuinely task-independent.

Same chance (1/104), same test-set windows as A1/A2, so numbers are directly comparable.

Send back `results/21_a2_vs_rest.json` and `figures/21_a2_vs_rest.pdf`.

**Runtime → L4 GPU.** Expected wall ~30-40 min.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery + baseline (~3 min)

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8
!python -m data.prefetch_direct --runs baseline --workers 8

## 5. Run experiment

Expected wall: ~30-40 min.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.21_a2_vs_rest --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = "experiments.21_a2_vs_rest"
ARGS = ['--all']
RESULTS = 'results/21_a2_vs_rest.json'
EXTRA_OUTPUTS = ['figures/21_a2_vs_rest.pdf']
TAG = "a2_vs_rest"
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS] + EXTRA_OUTPUTS}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — run cell did not finish. Re-run the run cell, then this cell.')
print(f'=== {RESULTS} ==='); print(open(RESULTS).read())


## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/21_a2_vs_rest.json')
files.download('figures/21_a2_vs_rest.pdf')
files.download(f'runs/{run_id}/meta.json')
